# 09 · A field changes its clothes

Can multiplication become a simple turn after we rearrange the same elements?

Start with coefficient pairs. Count the fibers of a finite-field norm. Choose a
generator inside its norm-one fiber, and use its powers to order every fiber.
The measured sizes then set the angular spacing. A multiplication that scatters
the coefficient grid becomes one cyclic step on every ring.

This develops the grid-to-fibers and finite-rotation experiments in
[Finite-Hermitian-Geometry](https://github.com/virgil-barnard/Finite-Hermitian-Geometry).
We work in **one quadratic field**; that project's product ring and CRT experiments
are separate. Every construction below uses Kaleion's existing integer operations.

**Try first:** run all cells, play the 3D motion, and rotate the camera when the
fibers rise into separate levels. Then change `P` or the phase generator. The rings
are a display of multiplication; their radii are not a finite-field metric.

In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import Code, Video, display
from kaleion import Motion, Workspace, param, vector
from kaleion.viewers.plotly import snapshot_figure
from kaleion.viewers.video import write_mp4

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'pyproject.toml').exists() and (p / 'src/kaleion').is_dir())
sys.path.insert(0, str(ROOT / 'notebooks'))
from lesson_views import COLORS, style, profiles, replay, save_figures
pio.renderers.default = 'plotly_mimetype+notebook'

OUTPUT = ROOT / 'build/notebooks/norm-fibers'
OUTPUT.mkdir(parents=True, exist_ok=True)
P = 7                         # Small teaching models: 3, 7, 11.
BETA = {3: 3, 7: 16, 11: 58}[P]  # Codes for i, 2+2i, and 3+5i.
INSPECT_NORM = 1
assert 0 <= INSPECT_NORM < P

## 1 · Declare the arithmetic before drawing it

For a prime $p\equiv3\pmod4$, the polynomial $X^2+1$ is irreducible and
$K=\mathbb F_p[i]$ has $p^2$ elements. We store $a+bi$ using the integer code $a+pb$,
with $0\le a,b<p$. The code is a label, **not arithmetic modulo $p^2$**.

\[
(a+bi)(c+di)=(ac-bd)+(ad+bc)i,\qquad
\overline{a+bi}=a-bi,\qquad N(a+bi)=a^2+b^2\pmod p.
\]

The small coordinate functions below compose those formulas from ordinary integer
and symbolic operations. They add no finite-field class or new evaluator opcode.
Their input convention is explicit so a later numerical domain can replace the
representation without taking responsibility for the camera or the history.

In [ ]:
from functools import reduce
from math import gcd, pi
from kaleion import Collection, F, choose, cos, sin
from quadratic_coordinates import (field_multiply, field_sum, field_conjugate,
                                   field_norm, hermitian_pair, element_label)

In [ ]:
import inspect
display(Code(inspect.getsource(field_multiply), language='python'))
print('Field:', f'F_{P}[i], i² = -1; generator β =', element_label(BETA, P))
assert field_multiply(P, P, P) == P-1
assert field_multiply(P, P, P) != (P*P) % (P*P)

In [ ]:
def norm_fibers(p):
    if p not in (3, 7, 11):
        raise ValueError("Use p in {3, 7, 11}; this field model requires p = 3 mod 4")
    elements = Collection.sequence(p*p, start=0).annotate(norm=field_norm(F.value, p))
    points = elements.arrange(F.value % p - (p-1)/2, F.value // p - (p-1)/2, 0)
    counts = points.count(by=F.norm).order_by(F.key)
    return points, counts

In [ ]:
points, counts = norm_fibers(P)
workspace = Workspace({'points': points, 'counts': counts})
assert not workspace.state.errors
state = workspace.state.results
assert state['counts'].values.tolist() == [1] + [P+1]*(P-1)
assert sum(state['counts'].values) == P*P

point_snapshot = state['points']
colors = [COLORS[(int(n)-1) % len(COLORS)] if n else '#64748b'
          for n in point_snapshot.fields['norm']]
coefficient_plot = style(go.Figure(go.Scatter(
    x=point_snapshot.positions[:,0], y=point_snapshot.positions[:,1], mode='markers+text',
    text=[element_label(z,P) for z in point_snapshot.values], textposition='top center',
    textfont=dict(size=10), marker=dict(size=13,color=colors),
    hovertext=[f'code {z} · norm {n}<br>{oid}' for z,n,oid in
               zip(point_snapshot.values,point_snapshot.fields['norm'],point_snapshot.ids)],
    hovertemplate='%{text}<br>%{hovertext}<extra></extra>')),
    f'{P*P} field elements · colors follow their norm fibers', height=570)
coefficient_plot.update_xaxes(title='a, centered for display',dtick=1)
coefficient_plot.update_yaxes(title='b, centered for display',dtick=1,scaleanchor='x')
coefficient_plot.show()
count_plot = profiles([state['counts']], ['Every residue has a declared fiber'],
                      keys=['key'], title='One zero · p+1 elements in every nonzero norm fiber')
count_plot.show()

## 2 · A count supplies the spacing; a generator supplies the order

The count alone cannot tell us which element comes next. Choose $\beta$ of order
$p+1$ with $N(\beta)=1$. Its powers enumerate that kernel. For each nonzero norm $n$,
choose the least **code** $r_n$ in the fiber, and write

\[
z=r_n\beta^k,\qquad 0\le k<c_n,\qquad
c_n=\#\{z:N(z)=n\}.
\]

This is an ordering convention, not a canonical finite-field angle. The short
loop below unrolls **symbolic products**, so the saved graph retains each power's
derivation. Fiber representatives come from selection, ordering, and gather.
Neither the powers nor their phase labels are fed back from evaluated arrays.

The point with code zero remains in the measurement and coefficient view. We move
only the nonzero elements: zero has no multiplicative phase.

In [ ]:
def phase_atlas(points, p, beta):
    if not isinstance(beta, int) or not 0 < beta < p*p or field_norm(beta, p) != 1:
        raise ValueError("Choose a nonzero coefficient code of norm one")
    orbit, value = [], 1
    for k in range(p+1):
        orbit.append(value)
        value = field_multiply(value, beta, p)
    if value != 1 or len(set(orbit)) != p+1:
        raise ValueError("The chosen phase generator must have order p+1")
    # Unroll a small declared number of symbolic products. No evaluated table
    # of phases is fed back as an anonymous literal.
    terms = [Collection.literal([1]).annotate(phase=0)]
    for k in range(1, p+1):
        terms.append(terms[-1].with_values(field_multiply(F.value, beta, p)).annotate(phase=k))
    powers = reduce(lambda a, b: a.concat(b), terms)
    representatives = [points.where(F.norm == n).select().order_by(F.value).gather([0])
                       for n in range(1, p)]
    anchors = reduce(lambda a, b: a.concat(b), representatives)
    atlas = Collection.grid(p-1, p+1).annotate(norm=F.i+1, phase=F.j)
    atlas = atlas.with_values(field_multiply(
        anchors.bind(on=F.norm, key=F.norm), powers.bind(on=F.phase, key=F.phase), p))
    return atlas, powers, anchors

def in_norm_fibers(source, atlas, counts, *, cylinder=False):
    # The caller supplies the norm attribute appropriate to its current values.
    marked = source.annotate(phase=atlas.bind(on=F.value, key=F.value, read=F.phase),
                             fiber_size=counts.bind(on=F.norm, key=F.key))
    angle = 2*pi*F.phase/F.fiber_size
    radius = 2 if cylinder else F.norm + 1
    return marked.arrange(radius*cos(angle), radius*sin(angle), F.norm if cylinder else 0)

In [ ]:
atlas, powers, anchors = phase_atlas(points, P, BETA)
nonzero = points.where(F.value != 0).select()
rings = in_norm_fibers(nonzero, atlas, counts)
cylinder = in_norm_fibers(nonzero, atlas, counts, cylinder=True)
atlas_workspace = Workspace({'atlas':atlas,'powers':powers,'anchors':anchors,
                             'counts':counts,'rings':rings,'cylinder':cylinder})
assert not atlas_workspace.state.errors
ar = atlas_workspace.state.results
assert set(ar['atlas'].values) == set(range(1,P*P))
assert len(set(ar['atlas'].values)) == P*P-1
assert set(ar['powers'].values) == {z for z in range(P*P) if field_norm(z,P)==1}
assert len(set(ar['rings'].ids)) == P*P-1
print('Measured fiber sizes:', state['counts'].values.tolist())
print('Powers of β:', [element_label(z,P) for z in ar['powers'].values])
print('Every nonzero code has one norm/phase address.')

## 3 · Rearrange, multiply, lift, undo

First move the coefficient grid into rings. Then replace each label $z$ by $\beta z$
and read its destination from the same atlas. Because the atlas uses powers of
$\beta$, every destination advances by one cyclic slot.

The smooth rotation is a recorded presentation path between exact endpoint
constructions. Its intermediate positions are not fractional field elements.
The cylinder then separates fibers in 3D. Its height is the chosen norm-residue
label; the angular spacing still comes from the measured cardinality.

In [ ]:
changed = nonzero.with_values(field_multiply(F.value,BETA,P)).annotate(norm=field_norm(F.value,P))
turned = in_norm_fibers(changed, atlas, counts)
raised = in_norm_fibers(changed, atlas, counts, cylinder=True)
motion_workspace = Workspace({'elements':nonzero})
gather_fibers = motion_workspace.set('elements', rings, motion=Motion())
t = param('time')
angle = 2*pi*t/(P+1)
rotation_path = Motion.custom(F.sx*cos(angle)-F.sy*sin(angle),
                              F.sx*sin(angle)+F.sy*cos(angle),F.sz)
rotate = motion_workspace.set('elements', turned, motion=rotation_path)
lift = motion_workspace.set('elements', raised, motion=Motion())
assert not motion_workspace.state.errors
initial = gather_fibers.start.results['elements']
final = lift.end.results['elements']
assert initial.ids == final.ids
assert [field_norm(z,P) for z in final.values] == initial.fields['norm'].tolist()
assert final.values.tolist() == [field_multiply(int(z),BETA,P) for z in initial.values]
motion_workspace.capture('A norm-one product is one cyclic step in a measured fiber atlas.')
unlift, unrotate, ungroup = (motion_workspace.undo(),motion_workspace.undo(),motion_workspace.undo())

samples, labels, plane_samples, plane_labels = [], [], [], []
stages = [('coefficient grid → fibers',gather_fibers),('multiply by β',rotate),
          ('lift the fibers',lift),('undo lift',unlift),('undo multiplication',unrotate),
          ('restore coefficients',ungroup)]
for stage, transition in stages:
    for fraction in np.linspace(0,1,25):
        frame = transition.frame('elements',float(fraction))
        caption = f'{stage} · {fraction:.0%}'
        samples.append(frame); labels.append(caption)
        if transition in (gather_fibers,rotate,unrotate,ungroup):
            # Explicit presentation projection; projected frames are never mathematical inputs.
            plane_samples.append(replace(frame,positions=frame.positions[:,:2]))
            plane_labels.append(caption)
for forward,reverse in ((gather_fibers,ungroup),(rotate,unrotate),(lift,unlift)):
    np.testing.assert_array_equal(forward.frame('elements',.25).positions,
                                  reverse.frame('elements',.75).positions)
np.testing.assert_allclose(samples[0].positions,samples[-1].positions,atol=1e-12,rtol=0)
by_id = {oid:COLORS[(int(n)-1)%len(COLORS)] for oid,n in zip(initial.ids,initial.fields['norm'])}
motion_plot = replay(samples,labels,title='The same field · grid, measured fibers, a cyclic turn',colors_by_id=by_id)
motion_plot.update_layout(scene_camera_eye=dict(x=1.65,y=1.65,z=1.3))
motion_plot.show()
cylinder_plot = snapshot_figure(ar['cylinder'],title='One measured fiber per level',show_values=False)
cylinder_plot.data[0].marker.color = [by_id[oid] for oid in ar['cylinder'].ids]
cylinder_plot.show()

## 4 · Change the coordinates while keeping every element fixed

For $\beta'=\beta^s$ with $\gcd(s,p+1)=1$, the **same** field element receives the
phase $k'=s^{-1}k\pmod{p+1}$. The following motion changes the atlas, not the labels.
Compare this with the preceding multiplication, which changed the labels themselves.

A non-generator is rejected: it repeats part of the orbit and cannot address a
whole fiber. That is a missing-correspondence problem, not a drawing problem.

In [ ]:
S = next(s for s in range(2,P+1) if gcd(s,P+1)==1)
alternative_beta = 1
for _ in range(S):
    alternative_beta = field_multiply(alternative_beta,BETA,P)
other_atlas,_,_ = phase_atlas(points,P,alternative_beta)
other_rings = in_norm_fibers(nonzero,other_atlas,counts)
coordinate_workspace = Workspace({'elements':rings})
recoordinate = coordinate_workspace.set('elements',other_rings,motion=Motion.arc(height=1.2,dimension=3))
before,after = recoordinate.start.results['elements'],recoordinate.end.results['elements']
assert before.ids == after.ids and before.values.tolist() == after.values.tolist()
assert after.fields['phase'].tolist() == [(pow(S,-1,P+1)*int(k))%(P+1) for k in before.fields['phase']]
restore_coordinates = coordinate_workspace.undo()
coordinate_samples,coordinate_labels = [],[]
for name,transition in [('change the phase generator',recoordinate),('restore the atlas',restore_coordinates)]:
    for fraction in np.linspace(0,1,25):
        coordinate_samples.append(transition.frame('elements',float(fraction)))
        coordinate_labels.append(f'{name} · {fraction:.0%}')
coordinate_plot = replay(coordinate_samples,coordinate_labels,
    title='A different generator · unchanged field elements',colors_by_id=by_id)
coordinate_plot.show()
try:
    phase_atlas(points,P,1)
except ValueError as error:
    print('Expected rejection:',error)
else:
    raise AssertionError('A repeated orbit must not define a complete atlas')

## 5 · Break the field assumption

Keep the same coefficient formulas but try $p=5$. Now $X^2+1=(X-2)(X+2)$,
so the quotient is not a field. The zero-norm fiber contains nonzero elements:
$(2+i)(2-i)=0$. We display all nine zero-norm elements, including zero.

Even in the valid field at $p=7$, a norm residue is not a Euclidean length.
$N(i)=N(2+2i)=1$, while $N(2+3i)=6$. Taking square roots of the least residues
would demand $\sqrt6\le2$, which fails. The ring pictures never claimed that metric.

In [ ]:
bad = Collection.sequence(25,start=0).annotate(norm=field_norm(F.value,5)).arrange(F.value%5,F.value//5)
zero_fiber = bad.where(F.norm==0)
bad_snapshot = zero_fiber.evaluate()
assert bad_snapshot.cardinality == 9
assert field_multiply(7,22,5)==0  # (2+i)(2-i)
bad_plot = snapshot_figure(bad_snapshot,title='Changing p to 5 · eight nonzero elements now have norm zero')
bad_plot.show()
assert [field_norm(z,7) for z in (7,16,23)] == [1,1,6]
print('Zero-norm codes at p=5:',bad_snapshot.source.values[bad_snapshot.mask].tolist())

## Why the fibers have equal size

The multiplicative group of $K$ is cyclic of order $p^2-1$. On nonzero elements,
$N(z)=z^{p+1}$ has a kernel of size $p+1$ and maps onto $\mathbb F_p^\times$.
Two elements have equal norm exactly when their ratio is in that kernel. Each
nonzero fiber is therefore one coset, with $p+1$ members; zero contributes the
remaining single element. This explains $p^2=1+(p-1)(p+1)$ for the whole family.

The coefficient formula also proves multiplicativity directly. In the atlas,
$\beta(r_n\beta^k)=r_n\beta^{k+1}$ explains the cyclic shift. A choice of radius,
camera, or animation duration does not enter either argument.

Background: [Conrad, *Finite Fields*, Theorem 5.7 and the appendix on cyclicity](https://kconrad.math.uconn.edu/blurbs/galoistheory/finitefields.pdf).
The examples and counterexamples adapt the norms and finite-rotation sections of
our [interactive Hermitian exposition](https://github.com/virgil-barnard/Finite-Hermitian-Geometry).

In [ ]:
count_snapshot = state['counts']
group_index = count_snapshot.fields['key'].tolist().index(INSPECT_NORM)
contributors = count_snapshot.contributor_ids(INSPECT_NORM)
source_index = {oid:i for i,oid in enumerate(point_snapshot.ids)}
explanation = {'norm':INSPECT_NORM,'count':len(contributors),'contributors':[
    {'occurrence':oid,'code':int(point_snapshot.values[source_index[oid]]),
     'element':element_label(point_snapshot.values[source_index[oid]],P)} for oid in contributors]}
assert len(contributors) == int(count_snapshot.values[group_index])
video_labels = [label.replace('β','beta').replace('→','to') for label in plane_labels]
movie = write_mp4(plane_samples,OUTPUT/'fiber-turn-and-undo.mp4',labels=video_labels,
                   title='Norm fibers: a 2D projection of grouping, multiplication, and undo',fps=24)
display(Video(str(movie),embed=True))
save_figures(OUTPUT,{'coefficient-grid':coefficient_plot,'fiber-counts':count_plot,
                    'grid-fibers-turn':motion_plot,'fiber-cylinder':cylinder_plot,
                    'generator-change':coordinate_plot,'reducible-counterexample':bad_plot})
for name,ws in [('measurements',workspace),('atlas',atlas_workspace),
                ('motion',motion_workspace),('coordinates',coordinate_workspace)]:
    # Compact whitespace only: retain every evaluated state, contributor, and path.
    payload=json.dumps(json.loads(ws.to_json()),separators=(',', ':'),allow_nan=False)
    (OUTPUT/f'{name}-workspace.json').write_text(payload)
    reopened=Workspace.from_json(payload)
    assert not reopened.state.errors
reopened=Workspace.from_json((OUTPUT/'motion-workspace.json').read_text())
replayed=reopened.redo()
np.testing.assert_array_equal(replayed.frame('elements',.25).positions,
                              gather_fibers.frame('elements',.25).positions)
(OUTPUT/'fiber-explanation.json').write_text(json.dumps(explanation,indent=2))
(OUTPUT/'checks.json').write_text(json.dumps({'p':P,'beta_code':BETA,
    'fiber_counts':list(map(int,count_snapshot.values)),'motion_frames':len(samples),
    'coordinate_frames':len(coordinate_samples),'video_frames':len(plane_samples),
    'p5_zero_fiber':bad_snapshot.cardinality},indent=2))
print('Saved six interactive views, a projected MP4, captured investigations, and a fiber explanation.')

The authoring choices exposed here are **arithmetic model, invariant, fiber,
generator, representative, correspondence, layout, and recorded path**. A future
interface should let us change those choices independently. A count supplies
the size of a fiber, but does not invent an order within it.

Next: [a point becomes a lens, and 28 points regroup](10_hermitian_partitions.ipynb).